# Mini Project YOLOv5: capybara และ raccoon

Notebook นี้เรียงลำดับตาม `MiniProject YOLOv5.docx.pdf` โดยใช้ Dataset จริง 2 classes จาก `data.yaml` ของผู้ใช้ ค่าที่ได้จากการ train และ evaluation ต้องอ่านจาก output หลังรันจริงเท่านั้น

**ก่อนรัน:** ใน Colab เลือก **Runtime > Change runtime type > Hardware accelerator > GPU > Save** แล้วรันทีละ Cell หาก Cell ใด error ให้แก้ก่อนรัน Cell ถัดไป


## Cell 1 — Clone YOLOv5 และติดตั้ง requirements

Cell นี้ทำอะไร: ดาวน์โหลด source code YOLOv5 ตาม Lab แล้วติดตั้ง dependencies ใน Colab


In [ ]:
%cd /content
!git clone https://github.com/ultralytics/yolov5.git
%cd /content/yolov5
!pip install -qr requirements.txt


## Cell 2 — ตรวจ PyTorch และ GPU

Cell นี้ทำอะไร: แสดงเวอร์ชัน PyTorch และชื่อ GPU; หยุดทันทีหาก Colab ยังไม่ได้ GPU


In [ ]:
import torch
from IPython.display import Image, display
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'กรุณาเลือก Runtime > Change runtime type > GPU แล้วรันใหม่'
print('GPU:', torch.cuda.get_device_name(0))


## Cell 3 — ทดลอง YOLOv5 กับภาพตัวอย่าง

Cell นี้ทำอะไร: ใช้ `yolov5s.pt` ตรวจภาพตัวอย่าง `data/images` ตาม Lab และแสดงภาพผลลัพธ์


In [ ]:
%cd /content/yolov5
!python detect.py --weights yolov5s.pt --img 640 --conf 0.25 --source data/images --project runs/detect --name lab-example --exist-ok
from pathlib import Path
example_image = Path('/content/yolov5/runs/detect/lab-example/zidane.jpg')
assert example_image.is_file(), f'ไม่พบผลตรวจตัวอย่าง: {example_image}'
display(Image(filename=str(example_image), width=600))


## Cell 4 — Clone Dataset จาก GitHub

Cell นี้ทำอะไร: clone repository แบบ Public ที่สร้างตาม Lab ลง `/content` และตรวจว่า directory ถูกสร้างจริง Repository นี้ต้องมี `train`, `valid`, `test`, `data.yaml` ไม่ใช่ ZIP อย่างเดียว


In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import subprocess

REPO_URL = 'https://github.com/miyu60/capybara-raccoon-yolov5-dataset.git'
parsed = urlparse(REPO_URL)
assert parsed.scheme == 'https' and parsed.netloc.lower() == 'github.com', 'ต้องเป็น URL https://github.com/...'
repo_name = Path(parsed.path.rstrip('/')).name.removesuffix('.git')
assert repo_name and repo_name not in ('.', '..')
DATASET_DIR = Path('/content') / repo_name
assert not DATASET_DIR.exists(), f'{DATASET_DIR} มีอยู่แล้ว; ตรวจหรือเปลี่ยนชื่อก่อน clone ใหม่'
subprocess.run(['git', 'clone', REPO_URL, str(DATASET_DIR)], check=True)
assert (DATASET_DIR / 'data.yaml').is_file(), 'Clone แล้วไม่พบ data.yaml ที่ root'
print('Dataset path:', DATASET_DIR)


## Cell 5 — ตรวจ Dataset ที่ Clone มา

Cell นี้ทำอะไร: อ่านชื่อ class จาก `data.yaml` จริง ตรวจจำนวนภาพ/label, คู่ชื่อไฟล์ และตัวเลขใน label ของทั้งสาม split


In [ ]:
import yaml

source_yaml = yaml.safe_load((DATASET_DIR / 'data.yaml').read_text())
print('data.yaml:', source_yaml)
assert source_yaml['nc'] == 2
assert source_yaml['names'] == ['capybara', 'raccoon']
for split in ('train', 'valid', 'test'):
    images_dir = DATASET_DIR / split / 'images'
    labels_dir = DATASET_DIR / split / 'labels'
    assert images_dir.is_dir() and labels_dir.is_dir(), f'{split}: directory ไม่ครบ'
    images = sorted(images_dir.glob('*.jpg'))
    labels = sorted(labels_dir.glob('*.txt'))
    assert len(images) == len(labels) > 0, f'{split}: จำนวน image/label ไม่เท่ากัน'
    assert {p.stem for p in images} == {p.stem for p in labels}, f'{split}: ชื่อ image/label ไม่ตรงกัน'
    objects = 0
    for label in labels:
        for line in label.read_text().splitlines():
            parts = line.split()
            assert len(parts) == 5, f'{label}: ไม่ใช่ bounding box 5 ค่า'
            class_id = int(parts[0])
            box = [float(x) for x in parts[1:]]
            assert 0 <= class_id < source_yaml['nc'] and all(0 <= x <= 1 for x in box)
            assert box[2] > 0 and box[3] > 0
            objects += 1
    print(f'{split}: images={len(images)}, labels={len(labels)}, objects={objects}')


## Cell 6 — ติดตั้ง Weights & Biases ตาม Lab

Cell นี้ทำอะไร: ติดตั้ง W&B สำหรับ dashboard ตามขั้น Lab แล้วให้เจ้าของบัญชีเข้าสู่ระบบเองใน Colab อย่าวาง API key ลงใน Notebook ที่แชร์ต่อ หากไม่ต้องการ dashboard ให้หยุดและตัดสินใจเรื่องขั้นนี้ก่อน train


In [ ]:
%pip install -q wandb
import wandb
wandb.login()


## Cell 7 — สร้าง YAML ใน YOLOv5/data และตั้ง `nc: 2` ใน model config

Cell นี้ทำอะไร: ใช้ path ที่ตรวจพบจากการ clone และชื่อ class จาก Dataset จริง สร้าง YAML ตาม Lab แล้วคัดลอก `yolov5s.yaml` มาตั้ง `nc: 2` โดยไม่แก้ไฟล์ต้นฉบับ


In [ ]:
import re
import shutil

LAB_YAML = Path('/content/yolov5/data/capybara-raccoon.yaml')
lab_data = {
    'train': str(DATASET_DIR / 'train' / 'images'),
    'val': str(DATASET_DIR / 'valid' / 'images'),
    'test': str(DATASET_DIR / 'test' / 'images'),
    'nc': source_yaml['nc'],
    'names': source_yaml['names'],
}
LAB_YAML.write_text(yaml.safe_dump(lab_data, sort_keys=False), encoding='utf-8')

source_model = Path('/content/yolov5/models/yolov5s.yaml')
LAB_MODEL = Path('/content/yolov5/models/yolov5s-capybara-raccoon.yaml')
model_text = source_model.read_text(encoding='utf-8')
model_text, changed = re.subn(r'(?m)^nc:\s*\d+', 'nc: 2', model_text, count=1)
assert changed == 1, 'ไม่พบค่า nc ใน yolov5s.yaml'
LAB_MODEL.write_text(model_text, encoding='utf-8')
print(LAB_YAML.read_text())
print('Model config:', LAB_MODEL)


## Cell 8 — ตรวจ path ก่อน train

Cell นี้ทำอะไร: ยืนยันว่า YAML ชี้ไปยังภาพทั้งสาม split จริง และ model config ใช้ 2 classes หากไม่ผ่านจะไม่ train


In [ ]:
confirmed = yaml.safe_load(LAB_YAML.read_text())
assert confirmed['nc'] == 2 and confirmed['names'] == ['capybara', 'raccoon']
for key in ('train', 'val', 'test'):
    path = Path(confirmed[key])
    assert path.is_dir() and len(list(path.glob('*.jpg'))) > 0, f'{key}: path ผิด {path}'
    print(key, path, len(list(path.glob('*.jpg'))), 'images')
assert re.search(r'(?m)^nc:\s*2\s*$', LAB_MODEL.read_text())
print('Path และ classes: PASS')


## Cell 9 — Train YOLOv5

Cell นี้ทำอะไร: train ด้วย GPU (`--device 0`), image size 416, 100 epochs และบันทึกผลภายใต้ `runs/train` ตาม Lab ชื่อ run มีเวลาเพื่อไม่ทับผลเดิม **รัน Cell นี้เมื่อทุก Cell ก่อนหน้าผ่านแล้วเท่านั้น**


In [ ]:
from datetime import datetime, timezone
RUN_NAME = 'capybara-raccoon-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
%cd /content/yolov5
!python train.py --img 416 --epochs 100 --data {LAB_YAML} --cfg {LAB_MODEL} --weights yolov5s.pt --device 0 --project runs/train --name {RUN_NAME}
RUN_DIR = Path('/content/yolov5/runs/train') / RUN_NAME
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
assert BEST_PT.is_file(), f'Train ไม่สำเร็จหรือไม่พบ best.pt: {BEST_PT}'
print('Training result:', RUN_DIR)
print('best.pt:', BEST_PT)


## Cell 10 — ตรวจผล Train และ metrics ที่เกิดขึ้นจริง

Cell นี้ทำอะไร: อ่าน `results.csv` ที่ YOLOv5 สร้างจริง แสดงบรรทัดสุดท้ายและไฟล์ผลลัพธ์ ไม่มีการเติมค่าตัวเลขล่วงหน้า


In [ ]:
import pandas as pd
results_csv = RUN_DIR / 'results.csv'
assert results_csv.is_file(), f'ไม่พบ {results_csv}'
results = pd.read_csv(results_csv)
results.columns = results.columns.str.strip()
metrics_columns = [c for c in results.columns if 'precision' in c or 'recall' in c or 'mAP' in c]
print('Epochs recorded:', len(results))
print(results[metrics_columns].tail(1).to_string(index=False))
for name in ('results.png', 'confusion_matrix.png'):
    path = RUN_DIR / name
    if path.is_file():
        display(Image(filename=str(path), width=800))


## Cell 11 — Validation และ Test ด้วย best.pt

Cell นี้ทำอะไร: รัน `val.py` กับ valid และ test แยกกันโดยใช้ `best.pt` ที่เพิ่งสร้าง ให้จด Precision, Recall, mAP จาก output จริง


In [ ]:
%cd /content/yolov5
!python val.py --weights {BEST_PT} --data {LAB_YAML} --img 416 --task val --device 0 --project runs/val --name {RUN_NAME}-valid
!python val.py --weights {BEST_PT} --data {LAB_YAML} --img 416 --task test --device 0 --project runs/val --name {RUN_NAME}-test


## Cell 12 — Prediction บน Test Dataset

Cell นี้ทำอะไร: ตรวจภาพ test ด้วย `best.pt`, บันทึกไฟล์ prediction และแสดงภาพตัวอย่าง ชื่อ class และ confidence ปรากฏบนกรอบในภาพ Confidence คือคะแนนความมั่นใจของการทำนาย ไม่ใช่ความถูกต้องของโมเดลโดยรวม


In [ ]:
%cd /content/yolov5
!python detect.py --weights {BEST_PT} --img 416 --conf 0.25 --source {DATASET_DIR / 'test' / 'images'} --device 0 --save-txt --save-conf --project runs/detect --name {RUN_NAME}-test
TEST_PRED_DIR = Path('/content/yolov5/runs/detect') / f'{RUN_NAME}-test'
test_predictions = sorted(TEST_PRED_DIR.glob('*.jpg'))
assert test_predictions, f'ไม่พบภาพ Prediction ใน {TEST_PRED_DIR}'
for prediction in test_predictions[:4]:
    print(prediction.name)
    display(Image(filename=str(prediction), width=500))
print('Saved predictions:', TEST_PRED_DIR)


## Cell 13 — อัปโหลดภาพ Capybara ใหม่และ Raccoon ใหม่

Cell นี้ทำอะไร: ให้ผู้ใช้เลือกภาพใหม่ **ครั้งละหนึ่งไฟล์** โดยครั้งแรกเป็น capybara และครั้งที่สองเป็น raccoon ภาพใหม่นี้ต้องไม่อยู่ใน train/valid/test


In [ ]:
from google.colab import files

NEW_IMAGES_DIR = Path('/content/yolov5/new_images')
NEW_IMAGES_DIR.mkdir(exist_ok=True)
print('เลือกภาพ Capybara ใหม่ 1 ภาพ')
capybara_upload = files.upload()
assert len(capybara_upload) == 1
print('เลือกภาพ Raccoon ใหม่ 1 ภาพ')
raccoon_upload = files.upload()
assert len(raccoon_upload) == 1
for name, data in [(next(iter(capybara_upload)), next(iter(capybara_upload.values()))),
                   (next(iter(raccoon_upload)), next(iter(raccoon_upload.values())))]:
    suffix = Path(name).suffix.lower()
    assert suffix in ('.jpg', '.jpeg', '.png'), f'รูปแบบภาพไม่รองรับ: {name}'
    (NEW_IMAGES_DIR / name).write_bytes(data)
print('New images:', [p.name for p in NEW_IMAGES_DIR.iterdir()])


## Cell 14 — ตรวจรูปใหม่ด้วย best.pt

Cell นี้ทำอะไร: รัน prediction กับรูปใหม่ทั้งสอง แสดง class/confidence บนภาพ และบันทึกผลใน `runs/detect`


In [ ]:
%cd /content/yolov5
!python detect.py --weights {BEST_PT} --img 416 --conf 0.25 --source {NEW_IMAGES_DIR} --device 0 --save-txt --save-conf --project runs/detect --name {RUN_NAME}-new
NEW_PRED_DIR = Path('/content/yolov5/runs/detect') / f'{RUN_NAME}-new'
new_predictions = sorted([p for p in NEW_PRED_DIR.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
assert len(new_predictions) >= 2, f'ผลรูปใหม่ไม่ครบ: {NEW_PRED_DIR}'
for prediction in new_predictions:
    print(prediction.name)
    display(Image(filename=str(prediction), width=600))
print('Saved predictions:', NEW_PRED_DIR)


## Cell 15 — ดาวน์โหลด best.pt และผลลัพธ์

Cell นี้ทำอะไร: ดาวน์โหลด `best.pt` และไฟล์ ZIP ผลการ train, validation และ prediction หลังรันจริง เก็บไฟล์ไว้เพราะ Colab runtime อาจถูกล้าง


In [ ]:
from google.colab import files
import shutil

assert BEST_PT.is_file()
files.download(str(BEST_PT))
archive_base = Path('/content') / f'{RUN_NAME}-results'
shutil.make_archive(str(archive_base), 'zip', root_dir='/content/yolov5/runs')
files.download(str(archive_base) + '.zip')


## ภาพ Screenshot ที่ควรถ่ายสำหรับรายงาน

1. โครงสร้าง `train`, `valid`, `test`, `data.yaml` บน GitHub
2. หน้า Colab ที่เลือก GPU และผล Cell ตรวจ PyTorch/GPU
3. ผลทดลอง YOLOv5 กับภาพตัวอย่าง
4. ผล clone Dataset และตารางจำนวนภาพ/label
5. เนื้อหา YAML ที่ระบุ 2 classes และ path จริง
6. คำสั่ง train และผล 100 epochs ที่รันครบ
7. `results.png`, Precision, Recall, mAP จาก output จริง
8. ภาพ prediction ของ valid/test และรูปใหม่อย่างละชนิด
9. path `best.pt` ที่ Cell train แสดงจริง
